# Experiment: Candidate Scoring Sensitivity

Objective: test whether the first thalassemia candidate ranking stays stable when safety, access, endpoint fit, or assay readiness gets more weight.

This notebook is research prioritization only. It is not medical advice.


In [ ]:
from __future__ import annotations

type Candidate = dict[str, float | str]
type Weights = dict[str, float]

BASE: Weights = {
    "endpoint": 0.30,
    "human": 0.25,
    "safety": 0.20,
    "access": 0.15,
    "assay": 0.10,
}
CANDIDATES: list[Candidate] = [
    {
        "lane": "hydroxyurea",
        "endpoint": 4,
        "human": 4,
        "safety": 3,
        "access": 5,
        "assay": 4,
    },
    {
        "lane": "sirolimus",
        "endpoint": 4,
        "human": 3,
        "safety": 2,
        "access": 3,
        "assay": 4,
    },
    {
        "lane": "epigenetic HbF targets",
        "endpoint": 5,
        "human": 2,
        "safety": 1,
        "access": 2,
        "assay": 5,
    },
    {
        "lane": "AMPKB1/autophagy",
        "endpoint": 4,
        "human": 2,
        "safety": 2,
        "access": 2,
        "assay": 4,
    },
    {
        "lane": "hepcidin-ferroportin",
        "endpoint": 3,
        "human": 3,
        "safety": 3,
        "access": 2,
        "assay": 3,
    },
    {
        "lane": "thalidomide class",
        "endpoint": 4,
        "human": 3,
        "safety": 1,
        "access": 2,
        "assay": 3,
    },
    {
        "lane": "curcuminoid analog HbF",
        "endpoint": 3,
        "human": 2,
        "safety": 2,
        "access": 3,
        "assay": 3,
    },
    {
        "lane": "EGCG / green-tea extract",
        "endpoint": 2,
        "human": 3,
        "safety": 2,
        "access": 4,
        "assay": 3,
    },
    {
        "lane": "ginger / 6-shogaol",
        "endpoint": 1,
        "human": 2,
        "safety": 3,
        "access": 3,
        "assay": 3,
    },
    {
        "lane": "melittin / bee venom",
        "endpoint": 1,
        "human": 1,
        "safety": 0,
        "access": 1,
        "assay": 2,
    },
]

In [ ]:
def weighted(candidate: Candidate, weights: Weights) -> float:
    """Calculate a weighted candidate score."""
    return round(sum(float(candidate[key]) * weights[key] for key in weights), 2)


def rank(weights: Weights) -> list[tuple[str, float, bool]]:
    """Rank candidates and mark lanes passing the hard safety gate."""
    rows = []
    for candidate in CANDIDATES:
        safe = (
            candidate["endpoint"] >= 3
            and candidate["human"] >= 2
            and candidate["safety"] >= 2
        )
        rows.append((str(candidate["lane"]), weighted(candidate, weights), bool(safe)))
    return sorted(rows, key=lambda row: row[1], reverse=True)


rank(BASE)[:6]

In [ ]:
PROFILES: dict[str, Weights] = {
    "base": BASE,
    "safety_first": {**BASE, "endpoint": 0.20, "safety": 0.35},
    "access_first": {**BASE, "endpoint": 0.20, "access": 0.30},
    "endpoint_first": {**BASE, "endpoint": 0.45, "access": 0.10},
    "assay_first": {**BASE, "human": 0.15, "assay": 0.35},
}
summary = {
    name: [row[0] for row in rank(weights)[:3]] for name, weights in PROFILES.items()
}
summary

## Interpretation

Hydroxyurea remains the stable affordable comparator. High-risk mechanism lanes can rank highly, but the hard safety gate keeps them out of patient-facing framing. Future assay data should replace these provisional scores.
